# 10b — Statistical Modeling

**The bridge between EDA and ML.** Statistical modeling tests whether patterns are real or noise, quantifies uncertainty, and provides the inferential rigor that consulting clients demand.

**Topics:** Hypothesis testing (t-test, chi-square, ANOVA, Mann-Whitney), A/B testing, confidence intervals, linear regression inference (OLS with statsmodels), time series decomposition (trend/seasonality/residual), ARIMA fundamentals.

**Reference:** [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html) | [statsmodels](https://www.statsmodels.org/stable/index.html)

**Allowed:** `pandas`, `numpy`, `scipy.stats`, `statsmodels`


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.datasets import fetch_openml

# Retail data for time series exercises
retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail = retail.dropna(subset=['CustomerID'])
retail['CustomerID'] = retail['CustomerID'].astype(int)
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
orders = retail[retail['Quantity'] > 0].copy()

# Weekly revenue time series
weekly_revenue = (
    orders.set_index('InvoiceDate')['Revenue']
    .resample('W').sum()
    .rename('revenue')
)

# Synthetic A/B test dataset
np.random.seed(42)
n_control = 5000
n_treatment = 4800
ab_test = pd.DataFrame({
    'group': ['control'] * n_control + ['treatment'] * n_treatment,
    'converted': (
        np.random.binomial(1, 0.12, n_control).tolist() +
        np.random.binomial(1, 0.135, n_treatment).tolist()
    ),
    'revenue': (
        np.where(np.random.binomial(1, 0.12, n_control),
                 np.random.lognormal(3.5, 0.8, n_control), 0).tolist() +
        np.where(np.random.binomial(1, 0.135, n_treatment),
                 np.random.lognormal(3.6, 0.8, n_treatment), 0).tolist()
    ),
    'user_age_days': np.random.exponential(90, n_control + n_treatment).astype(int)
})

# German credit for group tests
credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = credit_raw.copy()
credit['is_good'] = (credit['class'] == 'good').astype(int)
credit['credit_amount'] = pd.to_numeric(credit['credit_amount'], errors='coerce')
credit['duration'] = pd.to_numeric(credit['duration'], errors='coerce')
credit['age'] = pd.to_numeric(credit['age'], errors='coerce')

print(f"Weekly revenue: {len(weekly_revenue)} weeks")
print(f"A/B test: {len(ab_test)} users")
print(f"Credit: {len(credit)} records")

---
## Exercise 1 — Hypothesis Testing Toolkit

**Build a reusable testing framework.** In consulting, you run the same tests repeatedly on different datasets — having clean, composable functions is a differentiator.

Implement the following functions. Each must return a standardized result dict with: `test_name`, `statistic`, `p_value`, `is_significant` (at given alpha), `interpretation` (one-sentence string).

1. `two_sample_ttest(group_a, group_b, alpha=0.05, equal_var=False)` — Welch's t-test.
2. `mann_whitney_u(group_a, group_b, alpha=0.05)` — non-parametric alternative for non-normal data.
3. `chi_square_independence(contingency_table, alpha=0.05)` — test independence of two categoricals.
4. `one_way_anova(groups: list, alpha=0.05)` — test if 3+ group means differ.

Each `interpretation` string must reference the actual p-value and conclusion (e.g. `"p=0.023: significant difference between groups at alpha=0.05"`).

In [ ]:
def two_sample_ttest(group_a: pd.Series, group_b: pd.Series,
                     alpha: float = 0.05, equal_var: bool = False) -> dict:
    """
    Welch's t-test. Returns standardized result dict.
    """
    # YOUR CODE HERE
    pass

def mann_whitney_u(group_a: pd.Series, group_b: pd.Series,
                   alpha: float = 0.05) -> dict:
    """
    Mann-Whitney U test. Returns standardized result dict.
    """
    # YOUR CODE HERE
    pass

def chi_square_independence(contingency_table: pd.DataFrame,
                             alpha: float = 0.05) -> dict:
    """
    Chi-square test of independence.
    contingency_table: 2D crosstab DataFrame.
    Returns result dict with additional key 'cramers_v' (effect size).
    """
    # YOUR CODE HERE
    pass

def one_way_anova(groups: list, alpha: float = 0.05) -> dict:
    """
    One-way ANOVA for 3+ groups.
    groups: list of pd.Series.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- ASSERTIONS ---
required_keys = {'test_name', 'statistic', 'p_value', 'is_significant', 'interpretation'}

# t-test: good vs bad credit on credit_amount
good = credit[credit['is_good'] == 1]['credit_amount']
bad = credit[credit['is_good'] == 0]['credit_amount']
t_result = two_sample_ttest(good, bad)
assert set(t_result.keys()) == required_keys
assert isinstance(t_result['is_significant'], bool)
assert isinstance(t_result['interpretation'], str) and len(t_result['interpretation']) > 10

# Mann-Whitney
mw_result = mann_whitney_u(good, bad)
assert set(mw_result.keys()) == required_keys

# Chi-square: credit rating vs purpose
ctab = pd.crosstab(credit['class'], credit['purpose'])
chi_result = chi_square_independence(ctab)
assert 'cramers_v' in chi_result
assert 0 <= chi_result['cramers_v'] <= 1

# ANOVA: credit amount across employment duration groups
emp_groups = [credit[credit['employment'] == e]['credit_amount'].dropna()
              for e in credit['employment'].unique()]
anova_result = one_way_anova(emp_groups)
assert set(anova_result.keys()) == required_keys

print("✓ Exercise 1 passed")
print(f"t-test: {t_result['interpretation']}")
print(f"Chi-square: {chi_result['interpretation']}")
print(f"Cramer's V: {chi_result['cramers_v']:.4f}")

---
## Exercise 2 — A/B Test Analysis

**Business question:** Did the treatment (new checkout flow) significantly increase conversion and revenue?

Using `ab_test` DataFrame:

1. Write `ab_test_conversion(df, group_col, converted_col, alpha=0.05)` that:
   - Computes conversion rates for control and treatment.
   - Runs a **two-proportion z-test** (`statsmodels.stats.proportion.proportions_ztest`).
   - Computes **relative lift**: `(treatment_rate - control_rate) / control_rate * 100`.
   - Computes **95% confidence interval** for the lift.
   - Returns a dict: `control_rate`, `treatment_rate`, `lift_pct`, `lift_ci_95`, `z_stat`, `p_value`, `is_significant`, `sample_sizes`.

2. Write `ab_test_revenue(df, group_col, revenue_col, alpha=0.05)` that:
   - Tests mean revenue per user (including non-converters at $0).
   - Uses Mann-Whitney U (revenue is not normally distributed).
   - Computes effect size (Cohen's d using `(mean_diff / pooled_std)`).
   - Returns: `control_mean`, `treatment_mean`, `lift_pct`, `p_value`, `is_significant`, `effect_size`, `practical_significance` (bool: effect_size > 0.2).

3. Write `minimum_detectable_effect(baseline_rate, alpha=0.05, power=0.8)` that computes the MDE for a two-proportion test given desired power. Return MDE as a percentage point.

In [ ]:
def ab_test_conversion(df: pd.DataFrame, group_col: str,
                        converted_col: str, alpha: float = 0.05) -> dict:
    """
    Two-proportion z-test for conversion rate lift.
    """
    # YOUR CODE HERE
    pass

def ab_test_revenue(df: pd.DataFrame, group_col: str,
                    revenue_col: str, alpha: float = 0.05) -> dict:
    """
    Mann-Whitney U test for revenue per user.
    """
    # YOUR CODE HERE
    pass

def minimum_detectable_effect(baseline_rate: float,
                               alpha: float = 0.05,
                               power: float = 0.8) -> float:
    """
    Returns MDE as percentage points.
    Use normal approximation: z_alpha + z_beta threshold.
    """
    # YOUR CODE HERE
    pass

conv_result = ab_test_conversion(ab_test, 'group', 'converted')
rev_result = ab_test_revenue(ab_test, 'group', 'revenue')
mde = minimum_detectable_effect(baseline_rate=0.12)

In [ ]:
# --- ASSERTIONS ---
conv_keys = {'control_rate', 'treatment_rate', 'lift_pct', 'lift_ci_95',
             'z_stat', 'p_value', 'is_significant', 'sample_sizes'}
assert set(conv_result.keys()) == conv_keys
assert 0 < conv_result['control_rate'] < 1
assert len(conv_result['lift_ci_95']) == 2, "CI must be a (lower, upper) tuple"
assert conv_result['sample_sizes']['control'] == n_control
assert isinstance(conv_result['is_significant'], bool)

rev_keys = {'control_mean', 'treatment_mean', 'lift_pct', 'p_value',
            'is_significant', 'effect_size', 'practical_significance'}
assert set(rev_result.keys()) == rev_keys
assert isinstance(rev_result['practical_significance'], bool)

assert isinstance(mde, float)
assert 0 < mde < 20, "MDE should be a reasonable % for this baseline"

print("✓ Exercise 2 passed")
print(f"Conversion: control={conv_result['control_rate']:.2%}, "
      f"treatment={conv_result['treatment_rate']:.2%}, "
      f"lift={conv_result['lift_pct']:.1f}%, "
      f"significant={conv_result['is_significant']}")
print(f"Revenue: effect_size={rev_result['effect_size']:.4f}, "
      f"practical={rev_result['practical_significance']}")
print(f"MDE at 80% power: {mde:.2f} percentage points")

---
## Exercise 3 — Confidence Intervals & Effect Sizes

**Concept:** A p-value tells you if an effect exists. A confidence interval tells you how big it might be. Both are needed.

1. Write `bootstrap_ci(series, statistic_fn, n_bootstrap=1000, alpha=0.05, random_state=42)` that computes a bootstrap confidence interval for any statistic:
   - Resample with replacement `n_bootstrap` times.
   - Apply `statistic_fn` to each resample.
   - Return `(lower, upper, point_estimate)` using percentile method.

2. Apply `bootstrap_ci` to compute CIs for:
   - Mean `total_revenue` per customer (from `orders` grouped by CustomerID).
   - Median `credit_amount` for good vs bad credit customers.
   - Gini coefficient of revenue (reuse from 00b).

3. Write `cohens_d(group_a, group_b)` — standardized mean difference. Interpret: small (0.2), medium (0.5), large (0.8).

4. Write `sample_size_calculator(effect_size_d, alpha=0.05, power=0.8)` using the formula for two-sample t-test sample size.

In [ ]:
def bootstrap_ci(series: pd.Series, statistic_fn,
                  n_bootstrap: int = 1000,
                  alpha: float = 0.05,
                  random_state: int = 42):
    """
    Bootstrap confidence interval.
    Returns (lower, upper, point_estimate)
    """
    # YOUR CODE HERE
    pass

def cohens_d(group_a: pd.Series, group_b: pd.Series) -> dict:
    """
    Returns dict: d (effect size), magnitude ('small'/'medium'/'large')
    """
    # YOUR CODE HERE
    pass

def sample_size_calculator(effect_size_d: float,
                            alpha: float = 0.05,
                            power: float = 0.8) -> int:
    """
    Returns required sample size per group for two-sample t-test.
    Use: n = 2 * ((z_alpha/2 + z_beta) / effect_size_d)^2
    """
    # YOUR CODE HERE
    pass

# Apply
customer_revenue = orders.groupby('CustomerID')['Revenue'].sum()
ci_mean = bootstrap_ci(customer_revenue, np.mean)
ci_median = bootstrap_ci(customer_revenue, np.median)

d_result = cohens_d(good, bad)
n_required = sample_size_calculator(d_result['d'])

In [ ]:
# --- ASSERTIONS ---
assert len(ci_mean) == 3, "Must return (lower, upper, point_estimate)"
lower, upper, point = ci_mean
assert lower < point < upper, "Point estimate must be within CI"
assert lower > 0, "Revenue must be positive"

assert set(d_result.keys()) == {'d', 'magnitude'}
assert d_result['magnitude'] in {'small', 'medium', 'large'}
assert d_result['d'] >= 0  # absolute value

assert isinstance(n_required, int)
assert n_required > 0

print("✓ Exercise 3 passed")
print(f"Revenue mean CI: ${lower:,.0f} – ${upper:,.0f} (point: ${point:,.0f})")
print(f"Cohen's d (good vs bad credit): {d_result['d']:.4f} ({d_result['magnitude']})")
print(f"Sample size needed: {n_required} per group")

---
## Exercise 4 — OLS Regression with Statistical Inference

**statsmodels OLS gives you what sklearn doesn't:** p-values, confidence intervals, R², F-statistic, and diagnostic tests.

Using `credit` dataset, model `credit_amount` as a function of other variables:

1. Fit an OLS model: `credit_amount ~ duration + age + is_good` using `statsmodels.formula.api.ols`.
2. Extract and organize into `ols_summary_df`: coefficient name, `coef`, `std_err`, `t_stat`, `p_value`, `ci_lower`, `ci_upper`, `is_significant` (p < 0.05).
3. Report: `r_squared`, `adj_r_squared`, `f_statistic`, `f_p_value`, `n_observations`, `aic`, `bic`.
4. Run **diagnostic tests**:
   - Breusch-Pagan test for heteroscedasticity (`statsmodels.stats.diagnostic.het_breuschpagan`).
   - Durbin-Watson statistic for autocorrelation of residuals.
   - Jarque-Bera test for normality of residuals.
5. Return `model_report`: dict containing the summary df + all stats + diagnostic results.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera

def fit_ols_with_inference(df: pd.DataFrame, formula: str) -> dict:
    """
    Fit OLS model and return comprehensive model_report dict.
    Keys: ols_summary_df, r_squared, adj_r_squared, f_statistic,
          f_p_value, n_observations, aic, bic,
          heteroscedasticity, durbin_watson, jarque_bera
    """
    # YOUR CODE HERE
    pass

model_report = fit_ols_with_inference(
    credit, 'credit_amount ~ duration + age + is_good'
)

In [ ]:
# --- ASSERTIONS ---
required_keys = {'ols_summary_df', 'r_squared', 'adj_r_squared', 'f_statistic',
                 'f_p_value', 'n_observations', 'aic', 'bic',
                 'heteroscedasticity', 'durbin_watson', 'jarque_bera'}
assert set(model_report.keys()) == required_keys

df_coef = model_report['ols_summary_df']
assert list(df_coef.columns) == ['coef', 'std_err', 't_stat', 'p_value',
                                   'ci_lower', 'ci_upper', 'is_significant']
assert 0 < model_report['r_squared'] < 1
assert model_report['adj_r_squared'] <= model_report['r_squared']
assert model_report['n_observations'] == len(credit.dropna(subset=['credit_amount', 'duration', 'age']))
assert isinstance(model_report['durbin_watson'], float)
assert 0 <= model_report['durbin_watson'] <= 4

print("✓ Exercise 4 passed")
print(f"R²={model_report['r_squared']:.4f}, Adj R²={model_report['adj_r_squared']:.4f}")
print(f"F-stat={model_report['f_statistic']:.2f} (p={model_report['f_p_value']:.4f})")
print(f"Durbin-Watson={model_report['durbin_watson']:.3f}")
print(df_coef)

---
## Exercise 5 — Time Series Decomposition

**Business question:** How much of our revenue trend is structural growth vs seasonality vs noise?

Using `weekly_revenue`:

1. Run `seasonal_decompose` with `model='additive'`, `period=52` (annual seasonality in weekly data). Extract `trend`, `seasonal`, `resid` components.

2. Compute decomposition statistics:
   - `trend_strength`: `1 - var(resid) / var(trend + resid)` — what fraction of variance the trend explains.
   - `seasonal_strength`: `1 - var(resid) / var(seasonal + resid)`.
   - `noise_ratio`: `var(resid) / var(original)` — fraction that's unexplained.

3. Run **Augmented Dickey-Fuller** test on the original series and the detrended series. Report: `adf_statistic`, `p_value`, `is_stationary`.

4. Compute ACF and PACF values at lags 1–20. Return as `acf_pacf_df`: columns `lag`, `acf`, `pacf`.

5. Return all results in `decomp_report` dict.

In [ ]:
def decompose_time_series(series: pd.Series, period: int = 52) -> dict:
    """
    Full decomposition analysis.
    Returns decomp_report with keys:
      decomposition, trend_strength, seasonal_strength, noise_ratio,
      adf_original, adf_detrended, acf_pacf_df
    """
    # YOUR CODE HERE
    pass

decomp_report = decompose_time_series(weekly_revenue)

In [ ]:
# --- ASSERTIONS ---
required_keys = {'decomposition', 'trend_strength', 'seasonal_strength',
                 'noise_ratio', 'adf_original', 'adf_detrended', 'acf_pacf_df'}
assert set(decomp_report.keys()) == required_keys

assert 0 <= decomp_report['trend_strength'] <= 1
assert 0 <= decomp_report['seasonal_strength'] <= 1
assert 0 <= decomp_report['noise_ratio'] <= 1

for adf_key in ['adf_original', 'adf_detrended']:
    adf = decomp_report[adf_key]
    assert set(adf.keys()) == {'adf_statistic', 'p_value', 'is_stationary'}
    assert isinstance(adf['is_stationary'], bool)

acf_df = decomp_report['acf_pacf_df']
assert list(acf_df.columns) == ['lag', 'acf', 'pacf']
assert len(acf_df) == 20
assert acf_df['acf'].between(-1, 1).all()

print("✓ Exercise 5 passed")
print(f"Trend strength: {decomp_report['trend_strength']:.4f}")
print(f"Seasonal strength: {decomp_report['seasonal_strength']:.4f}")
print(f"Noise ratio: {decomp_report['noise_ratio']:.4f}")
print(f"Original series stationary: {decomp_report['adf_original']['is_stationary']}")

---
## Exercise 6 — ARIMA Forecasting

**Task:** Build and evaluate an ARIMA model for weekly revenue.

1. Write `find_arima_order(series, max_p=3, max_q=3, d=1)` that:
   - Fits ARIMA(p, d, q) for all combinations of p ∈ [0..max_p] and q ∈ [0..max_q].
   - Returns `aic_table`: DataFrame with `p`, `q`, `d`, `aic`, `bic`, sorted by AIC ascending.
   - Returns `best_order`: tuple `(p, d, q)` with lowest AIC.

2. Using `best_order`, fit a final ARIMA model on the first 90% of the series.

3. Forecast the remaining 10% (out-of-sample). Compute:
   - `forecast_df`: columns `date`, `actual`, `forecast`, `lower_ci`, `upper_ci`.
   - `mae`, `rmse`, `mape` (mean absolute percentage error) on the forecast period.

4. Return `arima_report`: dict with `best_order`, `aic_table`, `forecast_df`, `mae`, `rmse`, `mape`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

def find_arima_order(series: pd.Series, max_p: int = 3,
                      max_q: int = 3, d: int = 1):
    """
    Grid search over ARIMA(p,d,q) orders.
    Returns (aic_table DataFrame, best_order tuple)
    """
    # YOUR CODE HERE
    pass

def fit_and_forecast_arima(series: pd.Series, order: tuple,
                            train_pct: float = 0.9) -> dict:
    """
    Fit ARIMA on train split, forecast test split.
    Returns arima_report dict.
    """
    # YOUR CODE HERE
    pass

aic_table, best_order = find_arima_order(weekly_revenue, max_p=2, max_q=2, d=1)
arima_report = fit_and_forecast_arima(weekly_revenue, best_order)

In [ ]:
# --- ASSERTIONS ---
assert list(aic_table.columns) == ['p', 'q', 'd', 'aic', 'bic']
assert aic_table['aic'].is_monotonic_increasing, "AIC table must be sorted"
assert len(best_order) == 3
assert best_order[1] == 1  # d=1 as specified

assert set(arima_report.keys()) == {'best_order', 'aic_table', 'forecast_df', 'mae', 'rmse', 'mape'}
fc_df = arima_report['forecast_df']
assert list(fc_df.columns) == ['date', 'actual', 'forecast', 'lower_ci', 'upper_ci']
assert arima_report['mape'] > 0
assert (fc_df['lower_ci'] <= fc_df['forecast']).all()
assert (fc_df['forecast'] <= fc_df['upper_ci']).all()

print(f"✓ Exercise 6 passed")
print(f"Best ARIMA order: {best_order}")
print(f"Forecast MAE: {arima_report['mae']:,.0f} | RMSE: {arima_report['rmse']:,.0f} | MAPE: {arima_report['mape']:.2%}")
print(arima_report['forecast_df'].head())

---
## Exercise 7 — Post-Hoc Testing & Multiple Comparisons

**Problem:** When ANOVA says "at least one group is different", you need post-hoc tests to find *which* groups differ. And when you run many tests, you need to correct for multiple comparisons.

1. Using `credit`, test whether `credit_amount` differs across **all pairs** of `purpose` categories:
   - Run one-way ANOVA first.
   - If significant, run **Tukey's HSD** (`statsmodels.stats.multicomp.pairwise_tukeyhsd`).
   - Return `tukey_results_df`: group1, group2, mean_diff, p_adj, is_significant, reject.

2. Implement **Bonferroni correction** manually:
   - Given a list of p-values and original alpha, compute adjusted alpha = alpha / n_tests.
   - Return `bonferroni_df`: original_p, adjusted_alpha, is_significant_corrected.

3. Implement **Benjamini-Hochberg (FDR) correction**:
   - Sort p-values ascending, compute BH threshold for each rank.
   - Return `bh_df`: original_p, bh_threshold, is_significant_bh.

4. Apply both corrections to the pairwise p-values from step 1. Compare how many pairs survive each correction.

In [ ]:
def tukey_posthoc(df: pd.DataFrame, group_col: str,
                   value_col: str, alpha: float = 0.05):
    """
    ANOVA + Tukey HSD post-hoc test.
    Returns (anova_result dict, tukey_results_df)
    """
    # YOUR CODE HERE
    pass

def bonferroni_correction(p_values: list, alpha: float = 0.05) -> pd.DataFrame:
    """
    Returns bonferroni_df: original_p, adjusted_alpha, is_significant_corrected
    """
    # YOUR CODE HERE
    pass

def benjamini_hochberg(p_values: list, alpha: float = 0.05) -> pd.DataFrame:
    """
    FDR correction.
    Returns bh_df: original_p, bh_threshold, is_significant_bh
    """
    # YOUR CODE HERE
    pass

anova_res, tukey_df = tukey_posthoc(credit, 'purpose', 'credit_amount')

In [ ]:
# --- ASSERTIONS ---
assert list(tukey_df.columns) == ['group1', 'group2', 'mean_diff', 'p_adj', 'is_significant', 'reject']
assert (tukey_df['p_adj'] >= 0).all() and (tukey_df['p_adj'] <= 1).all()
assert tukey_df['is_significant'].dtype == bool

p_vals = tukey_df['p_adj'].tolist()
bonf_df = bonferroni_correction(p_vals)
assert list(bonf_df.columns) == ['original_p', 'adjusted_alpha', 'is_significant_corrected']
assert bonf_df['adjusted_alpha'].nunique() == 1  # same adjusted alpha for all

bh_df = benjamini_hochberg(p_vals)
assert list(bh_df.columns) == ['original_p', 'bh_threshold', 'is_significant_bh']

# BH should generally be less conservative than Bonferroni
assert bh_df['is_significant_bh'].sum() >= bonf_df['is_significant_corrected'].sum(), \
    "BH should retain at least as many as Bonferroni"

print("✓ Exercise 7 passed")
print(f"ANOVA: {anova_res['interpretation']}")
print(f"Tukey significant pairs: {tukey_df['is_significant'].sum()} / {len(tukey_df)}")
print(f"Bonferroni significant: {bonf_df['is_significant_corrected'].sum()}")
print(f"BH significant: {bh_df['is_significant_bh'].sum()}")

---
## Exercise 8 — Integrated Statistical Report

**Task:** The final deliverable — a unified statistical report function that would be the output of a data science engagement.

Write `full_statistical_report(df, target_col, group_col, time_col=None)` that:

1. **Descriptive block**: extended statistics for all numeric columns.
2. **Group comparison block**: for each numeric column, t-test and Mann-Whitney comparing `group_col` high vs low (split at median).
3. **Correlation block**: Pearson + Spearman correlations with `target_col`, ranked.
4. **OLS block**: OLS regression of `target_col` on all numeric predictors. Return key stats.
5. **Time series block** (if `time_col` provided): trend slope, ADF stationarity, top autocorrelation lag.

Return a dict with keys: `descriptive`, `group_tests`, `correlations`, `ols`, `time_series`.

Apply to `credit` with `target_col='credit_amount'`, `group_col='is_good'`.

In [ ]:
def full_statistical_report(df: pd.DataFrame, target_col: str,
                              group_col: str,
                              time_col: str = None) -> dict:
    """
    Comprehensive statistical report.
    Returns dict with keys: descriptive, group_tests, correlations, ols, time_series
    """
    # YOUR CODE HERE
    pass

stat_report = full_statistical_report(
    credit, target_col='credit_amount', group_col='is_good'
)

In [ ]:
# --- ASSERTIONS ---
assert set(stat_report.keys()) == {'descriptive', 'group_tests', 'correlations', 'ols', 'time_series'}
assert isinstance(stat_report['descriptive'], pd.DataFrame)
assert isinstance(stat_report['group_tests'], pd.DataFrame)
assert isinstance(stat_report['correlations'], pd.DataFrame)
assert isinstance(stat_report['ols'], dict)
assert stat_report['time_series'] is None  # No time_col provided

assert target_col not in stat_report['correlations'].index or True  # flexible
assert 'r_squared' in stat_report['ols']

corr_df = stat_report['correlations']
assert 'pearson' in corr_df.columns
assert 'spearman' in corr_df.columns

print("✓ Exercise 8 passed — Full statistical report generated")
print(f"\nOLS R²: {stat_report['ols']['r_squared']:.4f}")
print(f"\nTop correlates with {target_col}:")
print(stat_report['correlations'].head())